# Document-level embedding với Doc2Vec

## Goal

Từ model Doc2Vec đã train ở `01_train_doc2vec.ipynb`, tạo:

1. **Sentence-level vectors**: lấy trực tiếp từ `model.dv`, vì mỗi câu đã là một document lúc train.
2. **Document-level vector**: nối toàn bộ token của bài báo rồi dùng `model.infer_vector` — cùng mục tiêu với V2 của phoBERT (một vector duy nhất cho cả bài) nhưng theo cơ chế suy luận của Doc2Vec (gradient descent trên paragraph vector mới), không phải mean pooling.

## Setup

Chạy notebook từ thư mục `backend/`, sau khi đã chạy `01_train_doc2vec.ipynb` (cần `gensim` đã được cài trong cùng môi trường Python).

In [ ]:
from pathlib import Path

import numpy as np
import torch
from gensim.models.doc2vec import Doc2Vec

## Load models

In [ ]:
ARTIFACT_DIR = Path("artifacts")

dm_model = Doc2Vec.load(str(ARTIFACT_DIR / "doc2vec_dm.model"))
dbow_model = Doc2Vec.load(str(ARTIFACT_DIR / "doc2vec_dbow.model"))

print(f"PV-DM vector size: {dm_model.vector_size}")
print(f"PV-DBOW vector size: {dbow_model.vector_size}")

## Corpus

Phải giống hệt thứ tự câu dùng lúc train ở `01_train_doc2vec.ipynb`, vì sentence vector được tra theo tag là chỉ số dạng string (`"0"`, `"1"`, ...).

In [ ]:
# ruff: noqa: E501
segmented_sentences = [
    'Đặt vé từ TP HCM đi Singapore để công_tác , chị Hoàng_Loan , ở phường Xuân_Hoà , bất_ngờ vì mức giá lần đầu mua được kể từ sau đại_dịch " Năm_ngoái , chặng TP HCM - Singapore có lúc lên tới 3,2 triệu đồng một_chiều , còn năm nay tôi chỉ trả hơn 1,6 triệu đồng , đã gồm thuế , phí " , chị nói .',
    "Chiều về , vé cũng được áp_dụng mức giá khuyến_mại 19.000 đồng , nhưng sau khi cộng thuế , phí , tổng tiền chị Loan phải trả khoảng 2,4 triệu đồng .",
    "Theo chị , các khoản phí tại sân_bay Singapore cao hơn chiều bay từ Việt_Nam nên dù cùng giá vé niêm_yết , số tiền thực trả vẫn chênh_lệch đáng_kể .",
    "Tính cả hai chiều , chuyến đi Singapore của chị hết hơn 4 triệu đồng , giảm khoảng một_nửa so với cùng kỳ năm_ngoái .",
    "Sau Covid-19 , các đường_bay quốc_tế mất nhiều thời_gian để phục_hồi , trong khi nguồn cung chưa trở_lại như trước khiến giá luôn ở mức cao , nhất_là vào mùa du_lịch .",
    "Năm nay , nguồn cung tăng nhanh hơn , kéo_theo cạnh_tranh giữa các hãng và tạo thêm dư_địa giảm_giá .",
    "Mức giá chị Loan mua không phải trường_hợp cá_biệt .",
    "Khảo_sát các đường_bay từ TP HCM đi Singapore và Thái_Lan cho thấy mức giá khuyến_mại 19.000-90.000 đồng , chưa gồm thuế , phí chiếm đa_số các chặng bay trong tháng 8 và 9 .",
    "Sau khi cộng các khoản này , vé TP HCM - Singapore từ hơn 1,6 triệu đồng một_chiều , còn chặng TP HCM - Bangkok chưa đến 1,9 triệu đồng .",
    "Vé của một_số hãng hàng_không nước_ngoài trên cùng_đường bay hiện cao hơn khoảng 2-3 lần so với các hãng Việt_Nam .",
    "Từ Hà_Nội đi Singapore và Thái_Lan , giá vé của các hãng trong nước dao_động 2,6-3 triệu đồng một_chiều , đã gồm thuế , phí .",
    "Một_số ngày trong tháng 8 , mức thấp nhất còn hơn 2 triệu đồng .",
    "Với đường_bay TP HCM - Jakarta , giá cũng giảm nhưng mặt_bằng vẫn cao hơn Singapore và Thái_Lan .",
    "Nếu trước_đây vé khứ_hồi thường ở mức 7-10 triệu đồng , hiện giá thấp nhất khoảng 6,3 triệu đồng , đã gồm thuế , phí , tương_đương hơn 3 triệu đồng mỗi chiều .",
    "Mức giá cao hơn một phần do quãng đường_bay xa hơn .",
    "Các đường_bay từ Hà_Nội và TP HCM tới châu_Âu , Đông_Bắc_Á cũng giảm khoảng 10-15% so với trước .",
    "Giá đi xuống trong bối_cảnh nguồn cung hàng_không Việt_Nam tăng .",
    "Theo dữ_liệu dự_báo của Công_ty cung_cấp dữ_liệu hàng không OAG ( Anh ) , Việt_Nam có khoảng 7,3 triệu ghế cung_ứng trong tháng 8 , tăng 10% so với cùng kỳ năm_ngoái và đứng thứ hai Đông_Nam_Á , sau Indonesia .",
    "Trong khi tổng năng_lực khai_thác của thị_trường hàng_không Đông_Nam_Á tháng 8 chỉ tăng 0,8% so với cùng kỳ , nguồn cung của Việt_Nam tăng tới 10% .",
    "Trong đó , Vietnam_Airlines có khoảng 2,8 triệu ghế , tăng 8,2% , trong khi Vietjet khoảng 2,24 triệu ghế .",
    "Nguồn cung trên các đường_bay quốc_tế cũng được tăng_cường .",
    "Vietjet_Air cho biết , nâng tần_suất TP HCM - Kuala_Lumpur lên 7 chuyến mỗi tuần trong mùa cao_điểm , đồng_thời mở đường_bay TP HCM - Colombo từ ngày 18/8 .",
    "Hãng cũng chuẩn_bị khai_thác các đường_bay Hà_Nội - Almaty và Hà_Nội - Praha từ tháng 10 .",
    "Ông Hồng_Thanh , chủ một đại_lý vé máy_bay tại TP HCM , cho biết nguồn cung tăng và cạnh_tranh giữa các hãng là nguyên_nhân quan_trọng khiến giá vé quốc_tế hạ nhiệt .",
    "Các hãng phải tăng khuyến_mại , kích_cầu trong bối_cảnh sức_mua chưa phục_hồi như kỳ_vọng .",
    "Chi_phí nhiên_liệu cũng thuận_lợi hơn cho các hãng .",
    "Từ ngày 1/7 , Chính_phủ tiếp_tục kéo_dài thời_hạn áp_dụng thuế nhập_khẩu ưu_đãi , thuế bảo_vệ môi_trường và thuế_giá_trị gia_tăng với xăng_dầu , nhiên_liệu bay đến hết ngày 30/9/2026 , giúp giảm một phần chi_phí đầu_vào của các hãng hàng_không .",
    "Về nhu_cầu , thị_trường khách quốc_tế đến Việt_Nam tăng mạnh .",
    "Bảy tháng đầu năm , Việt_Nam đón gần 14 triệu lượt khách quốc_tế , tăng gần 14% so với cùng kỳ năm_ngoái .",
    "Riêng tháng 7 , lượng khách đạt khoảng 1,67 triệu lượt , trong đó đường_hàng không chiếm gần 83% .",
    "Nhu_cầu đi_lại quốc_tế tăng trong khi nguồn cung được bổ_sung khiến các hãng phải cạnh_tranh mạnh hơn để thu_hút khách .",
    "Đây cũng là một trong những yếu_tố kéo mặt_bằng giá xuống trong mùa hè năm nay .",
    "Không_chỉ quốc_tế , trước đó các hãng cũng liên_tục kích_cầu trên thị_trường nội_địa ngay giữa cao_điểm hè .",
    "Nhiều chương_trình đưa giá vé một_số chặng về mức 0 đồng hoặc vài chục nghìn đồng , chưa gồm thuế , phí .",
]

print(f"Number of sentences: {len(segmented_sentences)}")

In [ ]:
def tokenize(sentence: str) -> list[str]:
    """Tách theo khoảng trắng; corpus đã segment nên giữ nguyên từ ghép nối bằng '_' làm một token."""
    return sentence.split()

## 1. Sentence-level vectors

Paragraph vector cho từng câu đã được học sẵn trong `model.dv` lúc train, không cần infer lại.

In [ ]:
def sentence_vectors(model: Doc2Vec, num_sentences: int) -> torch.Tensor:
    """Tra paragraph vector đã học cho từng câu theo tag (chỉ số câu dạng string)."""
    vectors = np.stack([model.dv[str(index)] for index in range(num_sentences)])
    return torch.from_numpy(vectors).float()


dm_sentence_embeddings = sentence_vectors(dm_model, len(segmented_sentences))
dbow_sentence_embeddings = sentence_vectors(dbow_model, len(segmented_sentences))

print(f"PV-DM sentence embeddings shape: {dm_sentence_embeddings.shape}")
print(f"PV-DBOW sentence embeddings shape: {dbow_sentence_embeddings.shape}")

## 2. Document-level vector

Doc2Vec không có "mean pooling nhiều câu" như phoBERT V2; thay vào đó `infer_vector` chạy lại gradient descent trên token của toàn bộ input để suy ra một paragraph vector mới, nằm trong cùng không gian đã học lúc train.

In [ ]:
def infer_document_vector(
    model: Doc2Vec, sentences: list[str], epochs: int = 200
) -> torch.Tensor:
    """Suy luận vector cho cả bài bằng cách coi toàn bộ token nối lại là một paragraph mới."""
    all_tokens = [token for sentence in sentences for token in tokenize(sentence)]
    vector = model.infer_vector(all_tokens, epochs=epochs)
    return torch.tensor(vector, dtype=torch.float32)


dm_document_embedding = infer_document_vector(dm_model, segmented_sentences)
dbow_document_embedding = infer_document_vector(dbow_model, segmented_sentences)

print(f"PV-DM document embedding shape: {dm_document_embedding.shape}")
print(f"PV-DBOW document embedding shape: {dbow_document_embedding.shape}")

## Save

Lưu trong `notebook/Doc2Vec/artifacts/`, cùng chỗ với model đã train ở notebook trước, dùng lại đúng tên field (`embeddings` / `embedding`) như `phoBERT/03_document_embedding_v1.ipynb` và `.../v2.ipynb` để `03_similarity.ipynb` load được theo cùng pattern.

In [ ]:
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    {"num_sentences": len(segmented_sentences), "embeddings": dm_sentence_embeddings},
    OUTPUT_DIR / "dm_sentence_embeddings.pt",
)
torch.save(
    {"num_sentences": len(segmented_sentences), "embedding": dm_document_embedding},
    OUTPUT_DIR / "dm_document_embedding.pt",
)
torch.save(
    {"num_sentences": len(segmented_sentences), "embeddings": dbow_sentence_embeddings},
    OUTPUT_DIR / "dbow_sentence_embeddings.pt",
)
torch.save(
    {"num_sentences": len(segmented_sentences), "embedding": dbow_document_embedding},
    OUTPUT_DIR / "dbow_document_embedding.pt",
)

print("Saved sentence + document embeddings for PV-DM and PV-DBOW.")

## Next Steps

- `03_similarity.ipynb`: infer query vector cùng cách với `infer_document_vector`, rồi xếp hạng câu bằng cosine similarity, so sánh với ranking của PhoBERT.